# Silver Layer — Replenishment Engine

**SariMart Express — Automated Replenishment Demo (Microsoft Fabric)**

Reads the **Bronze** Delta tables and computes demand, safety stock, reorder points,
recommended order quantities, a 0-1 stockout **risk score**, and a **status**
(STOCKOUT / REORDER / OVERSTOCK / OK). It writes the unified table
`silver_replenishment_recommendations`, then produces one curated table per business question:

| Business question | Output table |
|-------------------|--------------|
| **UC1 - Which store/SKU is below its reorder point right now?** | `silver_uc1_reorder_actions` |
| **UC2 - Where in Metro Manila is stockout risk concentrated?** | `silver_uc2_stockout_risk_by_store` (heat-map source) |
| **UC3 - Which items are overstocked / tying up cash?** | `silver_uc3_overstock` |

**Key design choices (adapted from the reference engine):**
- Current stock comes from `fact_inventory_snapshot` (the authoritative stock position), **not** from sales.
- `date_key` (yyyymmdd INT) is converted to a real date; the demand window is anchored to the latest sales date.
- Demand uses real recent sales where history is sufficient, and falls back to a planning estimate
  (`base_demand x traffic tier`) for sparse store/SKU combinations - so reorder points are always sensible.
- `available_qty = on_hand + in_transit` drives the reorder decision; `on_hand` drives stockout risk.

> **Prerequisite:** run the **Bronze** notebook first. Attach `LH_Retail`, then **Run all**.

In [ ]:
# 1. Configuration & Silver sources (gold outputs are written to golddb)
from pyspark.sql import functions as F

Z                    = 1.65   # 95% service level (safety stock)
REVIEW_DAYS          = 7      # reorder review cycle
LOOKBACK_DAYS        = 90     # window for sales-based demand
OVERSTOCK_COVER_DAYS = 14     # days-of-cover above this = overstock

SILVER_SCHEMA = "silverdb"
GOLD_SCHEMA = "golddb"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")

sales = spark.table(f"{SILVER_SCHEMA}.sales_dim")
inv   = spark.table(f"{SILVER_SCHEMA}.inventory_snapshot_dim")
prod  = spark.table(f"{SILVER_SCHEMA}.product_dim")
store = spark.table(f"{SILVER_SCHEMA}.store_dim")
cat   = spark.table(f"{SILVER_SCHEMA}.category_dim")
print("Silver sources loaded")

In [ ]:
# 2. Demand signal - average daily demand & variability from recent sales
#    date_key is yyyymmdd (INT) -> convert to a real date, then anchor the window
#    to the latest sales date in the data (robust for a synthetic dataset).
sales = sales.withColumn("sdate", F.to_date(F.col("date_key").cast("string"), "yyyyMMdd"))
max_date = sales.agg(F.max("sdate")).first()[0]
recent = sales.filter(F.col("sdate") > F.date_sub(F.lit(max_date), LOOKBACK_DAYS))

# daily units per store x product, then a true daily demand rate over the window
daily = (recent.groupBy("store_key", "product_key", "sdate")
               .agg(F.sum("units_sold").alias("day_units")))
demand = (daily.groupBy("store_key", "product_key")
               .agg((F.sum("day_units") / F.lit(LOOKBACK_DAYS)).alias("sales_demand"),
                    F.stddev_samp("day_units").alias("sales_std"),
                    F.countDistinct("sdate").alias("active_days")))
print("demand window up to", max_date, "| store-SKU demand rows:", demand.count())

In [ ]:
# 3. Latest inventory position (most recent snapshot date)
latest_dk = inv.agg(F.max("date_key")).first()[0]
inv_latest = inv.filter(F.col("date_key") == latest_dk)
print("snapshot date_key:", latest_dk, "| store-SKU rows:", inv_latest.count())

In [ ]:
# 4. Replenishment engine - join inventory + demand + product + store + category
df = (inv_latest
      .join(demand, ["store_key", "product_key"], "left")
      .join(prod.select("product_key","product_name","category_key","unit_cost",
                        "case_pack","lead_time_days","base_demand"), "product_key")
      .join(store.select("store_key","store_name","city","latitude","longitude","traffic_tier"), "store_key")
      .join(cat.select("category_key","category_name","is_perishable"), "category_key"))

# store foot-traffic multiplier -> model-based demand fallback for sparse SKUs
df = df.withColumn("tier_mult",
        F.when(F.col("traffic_tier")==3, 1.6)
         .when(F.col("traffic_tier")==2, 1.0).otherwise(0.7))

df = (df
  # demand: use real sales where there's enough history, else a planning estimate
  .withColumn("avg_daily_demand",
        F.when((F.col("active_days") >= 3) & (F.col("sales_demand") > 0), F.round(F.col("sales_demand"), 2))
         .otherwise(F.round(F.col("base_demand") * F.col("tier_mult"), 2)))
  .withColumn("demand_std",
        F.coalesce(F.col("sales_std"), F.col("avg_daily_demand") * F.lit(0.30)))
  .withColumn("available_qty", F.col("on_hand_qty") + F.col("units_in_transit"))
  # safety stock = Z * sigma * sqrt(lead time)
  .withColumn("safety_stock",
        F.expr(f"ceil({Z} * demand_std * sqrt(lead_time_days))"))
  # reorder point = demand over lead time + safety stock
  .withColumn("reorder_point",
        F.expr("ceil(avg_daily_demand * lead_time_days + safety_stock)"))
  .withColumn("days_of_cover",
        F.when(F.col("avg_daily_demand") > 0,
               F.round(F.col("on_hand_qty")/F.col("avg_daily_demand"), 1)).otherwise(F.lit(999.0)))
  .withColumn("needs_reorder", F.col("available_qty") <= F.col("reorder_point"))
  # target stock = cover (lead time + review cycle) of demand, then round up to case pack
  .withColumn("target_qty",
        F.expr(f"ceil(avg_daily_demand * (lead_time_days + {REVIEW_DAYS}) + safety_stock)"))
  .withColumn("raw_order_qty", F.greatest(F.col("target_qty") - F.col("available_qty"), F.lit(0)))
  .withColumn("recommended_order_qty",
        F.expr("ceil(raw_order_qty / case_pack) * case_pack"))
  # risk score 0..1 - how far below ROP on-hand is (drives heat map + ranking)
  .withColumn("risk_score",
        F.when(F.col("reorder_point") > 0,
               F.round(F.least(F.lit(1.0),
                   (F.col("reorder_point")-F.col("on_hand_qty"))/F.col("reorder_point")), 3))
         .otherwise(F.lit(0.0)))
  .withColumn("risk_score", F.greatest(F.col("risk_score"), F.lit(0.0)))
  .withColumn("est_order_cost", F.round(F.col("recommended_order_qty")*F.col("unit_cost"), 2))
  .withColumn("status",
        F.when(F.col("on_hand_qty")==0, "STOCKOUT")
         .when(F.col("needs_reorder"), "REORDER")
         .when(F.col("days_of_cover") > OVERSTOCK_COVER_DAYS, "OVERSTOCK")
         .otherwise("OK")))

silver = df.select(
    "date_key","store_key","store_name","city","latitude","longitude",
    "product_key","product_name","category_name","is_perishable","unit_cost",
    "on_hand_qty","units_in_transit","available_qty",
    "avg_daily_demand","demand_std","safety_stock","reorder_point","days_of_cover",
    "needs_reorder","recommended_order_qty","est_order_cost","risk_score","status")

(silver.write.mode("overwrite").option("overwriteSchema","true")
       .format("delta").saveAsTable(f"{GOLD_SCHEMA}.replenishment_recommendations"))
print(f"✓ {GOLD_SCHEMA}.replenishment_recommendations written:", silver.count(), "rows")

In [ ]:
# 5. Sanity check - distribution of statuses across the estate
silver.groupBy("status").count().orderBy("status").show()

## UC1 - Reorder Alerting
*Which store + SKU combinations are below their reorder point right now?* Ranked by urgency.

In [ ]:
uc1 = (silver.filter(F.col("needs_reorder"))
       .select("store_name","city","product_name","category_name",
               "on_hand_qty","units_in_transit","reorder_point",
               "recommended_order_qty","est_order_cost","risk_score","status")
       .orderBy(F.desc("risk_score"), F.desc("est_order_cost")))
(uc1.write.mode("overwrite").option("overwriteSchema","true")
    .format("delta").saveAsTable(f"{GOLD_SCHEMA}.uc1_reorder_actions"))
print("Items to reorder:", uc1.count(),
      "| Est. total order cost:", round(uc1.agg(F.sum("est_order_cost")).first()[0] or 0, 2))
uc1.show(15, truncate=False)

## UC2 - Stockout-Risk Heat Map
*Where in Metro Manila is stockout risk concentrated today?* Aggregated per store with lat/long -
this is the source table the Azure Maps **heat map** plots, weighted by `total_stockout_risk`.

In [ ]:
uc2 = (silver.groupBy("store_key","store_name","city","latitude","longitude")
       .agg(F.round(F.sum("risk_score"),2).alias("total_stockout_risk"),
            F.round(F.avg("risk_score"),3).alias("avg_risk_score"),
            F.sum(F.col("needs_reorder").cast("int")).alias("reorder_items"),
            F.sum((F.col("status")=="STOCKOUT").cast("int")).alias("stockout_items"))
       .orderBy(F.desc("total_stockout_risk")))
(uc2.write.mode("overwrite").option("overwriteSchema","true")
    .format("delta").saveAsTable(f"{GOLD_SCHEMA}.uc2_stockout_risk_by_store"))
print("Heat-map source ready - plot latitude/longitude weighted by total_stockout_risk")
uc2.show(30, truncate=False)

## UC3 - Overstock / Slow-Mover Detection
*Which items are overstocked, tying up cash or nearing expiry?* Perishable items are flagged.

In [ ]:
uc3 = (silver.filter(F.col("status")=="OVERSTOCK")
       .withColumn("tied_up_cash", F.round(F.col("on_hand_qty")*F.col("unit_cost"), 2))
       .select("store_name","city","product_name","category_name","is_perishable",
               "on_hand_qty","days_of_cover","tied_up_cash")
       .orderBy(F.desc("tied_up_cash")))
(uc3.write.mode("overwrite").option("overwriteSchema","true")
    .format("delta").saveAsTable(f"{GOLD_SCHEMA}.uc3_overstock"))
print("Overstock SKUs:", uc3.count(),
      "| Cash tied up:", round(uc3.agg(F.sum("tied_up_cash")).first()[0] or 0, 2),
      "| Perishable at risk:", uc3.filter(F.col("is_perishable")).count())
uc3.show(15, truncate=False)

## Notes

- **Stock vs demand sources:** stock position is read from `fact_inventory_snapshot` (latest snapshot date);
  demand velocity is derived from `fact_sales` over the last 90 days. This is the correct separation -
  inventory is a *periodic snapshot* fact, sales is a *transaction* fact.
- **Sparse-history fallback:** store/SKU pairs with fewer than 3 active sales days use a planning estimate
  (`base_demand x traffic tier`) so reorder points never collapse to zero on thin synthetic data.
- **Semi-additive stock:** only the latest snapshot is used; on-hand is never summed across dates.
- **Outputs answer the three business questions directly** - `silver_uc1_reorder_actions`,
  `silver_uc2_stockout_risk_by_store`, and `silver_uc3_overstock` feed the Power BI report pages
  (action list, Azure Maps heat map, and overstock report).
- **Tuning knobs:** `Z` (service level), `REVIEW_DAYS`, `LOOKBACK_DAYS`, and `OVERSTOCK_COVER_DAYS` at the top.